# Course 4 Lab — Agent Identity and Delegated Authority

**Scenario:** A procurement agent may create one purchase order for approved laptops. A vendor-research sub-agent may only read vendor information.

The trusted application—not the language model—binds the authenticated human, logical agent version, attested workload, tenant, and task. A policy enforcement point verifies and atomically consumes narrow delegated authority before any effect.

## Success criteria

- valid, bounded authority permits the intended operation;
- forged identity, wrong audience/resource/task, excessive amount, stale identity, tampered token, replay mutation, and revoked lineage fail closed;
- every child grant is narrower across actions, resources, vendors, amount, calls, lifetime, audience transition, and depth;
- evaluation reports its exact labelled population and safety errors.

## Safety and reproducibility

This notebook is offline, credential-free, deterministic, and uses synthetic identities. The fixed Ed25519 key is public test material. The JWT is a teaching artifact—not an OAuth server, production access-token profile, legal proof, or non-repudiation guarantee.

## 1. Environment and architecture

The reusable implementation uses **Pydantic** contracts, **PyJWT**, and **cryptography**. OpenFGA, Cedar/Verified Permissions, OPA, SPIFFE/SPIRE, and an OAuth authorization server are production integration choices discussed later; they are not required to run the lab.

```text
authenticated human session + workload attestation + agent registry
                              ↓
                    trusted task context
                              ↓
          approved intent → narrow signed task grant
                              ↓
    proposed operation → PEP → verify → authorize + atomic consume
                              ↓
                    effect adapter + audit digest
```

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from datetime import timedelta
from importlib.metadata import version
from pathlib import Path
import sys

COURSE_DIR = Path("curriculum/beginner/04-agent-identity-and-delegated-authority").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

from lab import (
    ALLOWED_AUDIENCE_TRANSITIONS,
    REFERENCE_TIME,
    AuthorizationRequest,
    DecisionOutcome,
    DelegationConstraints,
    authorize_token,
    attenuate_grant,
    build_demo_authority,
    demo_context,
    demo_directory,
    demo_request,
    run_evaluation,
    token_digest,
    token_exchange_request,
)

print({name: version(name) for name in ("pydantic", "PyJWT", "cryptography", "openfga_sdk")})
print("Reference time:", REFERENCE_TIME.isoformat())

## 2. Baseline: authentication or a scope string is not enough

This intentionally unsafe baseline trusts identity fields supplied with the request and checks only a coarse scope. It cannot prove which workload is calling, bind the action to an approved task/resource, enforce an amount, revoke a delegation chain, or prevent concurrent reuse.

In [ ]:
def unsafe_scope_only_authorize(request: dict) -> bool:
    return request.get("claimed_actor") == "agent:procurement" and "purchase.write" in request.get("scopes", [])

unsafe_request = {
    "claimed_actor": "agent:procurement",
    "claimed_workload": "spiffe://attacker.example/forged",
    "scopes": ["purchase.write"],
    "resource": "department:finance",
    "amount_cents": 2_500_000,
}
assert unsafe_scope_only_authorize(unsafe_request) is True
print("Unsafe baseline allowed forged, over-scoped request:", unsafe_scope_only_authorize(unsafe_request))

The baseline's `True` is a failure, not a success. A model or request body can propose identity, but the application must derive identity and scope from authenticated state.

## 3. Bind the human, agent, and workload

The directory contains distinct principals and explicit agent-to-workload registrations. The workload evidence represents the output of an attestation system such as SPIRE; a string that merely looks like a SPIFFE ID is not proof.

In [ ]:
directory = demo_directory()
context = demo_context()
print(context.model_dump(mode="json"))

assert context.subject_id == "human:user-123"
assert context.actor_id == "agent:procurement"
assert context.actor_version == "1.0.0"
assert context.workload_id == "spiffe://example.com/prod/procurement"

Identity binding checks tenant equality, trust domain, current attestation time, registered selectors, active principals, and the exact agent/workload association. Course 5 will go deeper on the authorization models used after authentication.

## 4. Bind a signed grant to approved business intent

The root grant is derived from an approved intent and a trusted task context. Requested actions, resources, vendors, amount, and calls cannot exceed that intent. The lab signs a deterministic JWT so learners can inspect verification behavior without credentials.

In [ ]:
context, grant, token, ledger = build_demo_authority()
verified = __import__("lab").verify_training_token(token, audience=grant.audience, now=REFERENCE_TIME)

print({
    "grant_id": grant.grant_id,
    "subject": grant.subject_id,
    "actor": f"{grant.actor_id}@{grant.actor_version}",
    "workload": grant.workload_id,
    "task": grant.task_id,
    "audience": grant.audience,
    "actions": sorted(grant.actions),
    "resources": sorted(grant.resources),
    "expires_at": grant.expires_at.isoformat(),
    "token_digest": token_digest(token),
})
assert verified == grant

The verifier pins the header, EdDSA algorithm, test key, issuer, audience, required claims, temporal validity, and structured authorization details. Production deployments should use an approved authorization server/STS, managed signing keys, rotation, discovery, and sender-constrained tokens where appropriate.

## 5. Policy enforcement point: test every request dimension

The operation contains no caller-controlled subject, actor, workload, tenant, or task fields. Those come from `TrustedTaskContext`. Each case gets a fresh ledger so the one-call grant does not obscure the reason under test.

In [ ]:
cases = (
    ("allowed", {}, True),
    ("amount over limit", {"amount_cents": 500_001}, False),
    ("wrong vendor", {"vendor_id": "vendor-unknown"}, False),
    ("wrong resource", {"resource": "department:finance"}, False),
    ("undelegated action", {"action": "payment:issue"}, False),
)
rows = []
for index, (name, changes, expected) in enumerate(cases, start=1):
    case_context, _, case_token, case_ledger = build_demo_authority()
    request = demo_request(operation_id=f"OP-MATRIX-{index}").model_copy(update=changes)
    decision = authorize_token(case_ledger, case_token, request, case_context, now=REFERENCE_TIME)
    actual = decision.outcome is DecisionOutcome.ALLOW
    rows.append((name, expected, actual, decision.reason_codes))
    assert actual is expected

for row in rows:
    print(row)

One allow and four specific denials show why a valid signature is necessary but insufficient. Authorization still evaluates exact task authority and current server-side lifecycle state.

## 6. Idempotency and replay mutation

A stable operation ID identifies one logical effect. An identical retry receives the same decision without consuming another call. Reusing that ID for changed arguments is denied.

In [ ]:
context, _, token, ledger = build_demo_authority()
request = demo_request()
first = authorize_token(ledger, token, request, context, now=REFERENCE_TIME)
retry = authorize_token(ledger, token, request, context, now=REFERENCE_TIME)
collision = authorize_token(
    ledger,
    token,
    request.model_copy(update={"amount_cents": 100_000}),
    context,
    now=REFERENCE_TIME,
)
print(first.outcome, retry.replayed_decision, collision.reason_codes)
assert first.outcome is DecisionOutcome.ALLOW
assert retry.decision_id == first.decision_id and retry.replayed_decision
assert collision.reason_codes == ("operation_id_reused_with_different_request",)

An authorization decision is not itself proof that an external side effect occurred. A production effect adapter must use the same operation ID, persist result state, and reconcile an unknown outcome before retrying.

## 7. Atomic call-limit experiment

The prototype notebook used a check-then-increment dictionary, which can race. The ledger holds one lock across validation and consumption. Eight concurrent unique operations compete for a one-call grant; exactly one may succeed.

In [ ]:
context, _, token, ledger = build_demo_authority()

def concurrent_attempt(index: int):
    return authorize_token(
        ledger,
        token,
        demo_request(operation_id=f"OP-RACE-{index}"),
        context,
        now=REFERENCE_TIME,
    )

with ThreadPoolExecutor(max_workers=8) as pool:
    race_decisions = list(pool.map(concurrent_attempt, range(8)))

allowed_count = sum(item.outcome is DecisionOutcome.ALLOW for item in race_decisions)
denied_count = sum(item.outcome is DecisionOutcome.DENY for item in race_decisions)
print({"population": 8, "allowed": allowed_count, "denied": denied_count})
assert (allowed_count, denied_count) == (1, 7)

The in-process lock makes this deterministic for the lab. Production needs a transactional database, compare-and-swap/row lock, or another atomic store shared by all enforcement replicas.

## 8. Sub-agent delegation must attenuate every dimension

The research agent receives a different registered workload identity, a downstream vendor-service audience, only `vendor:read`, the same resource, no spending authority, one approved vendor, a shorter lifetime, and no further delegation depth.

In [ ]:
_, parent, parent_token, ledger = build_demo_authority()
child_context = demo_context(research_agent=True)
child_constraints = DelegationConstraints(
    max_amount_cents=0,
    allowed_vendor_ids=frozenset({"vendor-acme"}),
    max_calls=1,
)
child, child_token = attenuate_grant(
    parent_token,
    child_context,
    ledger,
    parent_audience=parent.audience,
    grant_id="GRANT-CHILD-001",
    audience="https://api.example.com/vendors",
    actions={"vendor:read"},
    resources={"department:data-ai"},
    constraints=child_constraints,
    expires_at=REFERENCE_TIME + timedelta(minutes=5),
    now=REFERENCE_TIME,
)
print({
    "parent": child.parent_grant_id,
    "depth": child.depth,
    "actor": child.actor_id,
    "audience": child.audience,
    "actions": sorted(child.actions),
    "max_amount_cents": child.constraints.max_amount_cents,
})
assert child.actions < parent.actions
assert child.expires_at < parent.expires_at

Now inject privilege amplification. A child that adds `payment:issue` is rejected before a token is issued.

In [ ]:
try:
    attenuate_grant(
        parent_token,
        child_context,
        ledger,
        parent_audience=parent.audience,
        grant_id="GRANT-CHILD-AMPLIFIED",
        audience="https://api.example.com/vendors",
        actions={"vendor:read", "payment:issue"},
        resources={"department:data-ai"},
        constraints=child_constraints,
        expires_at=REFERENCE_TIME + timedelta(minutes=5),
        now=REFERENCE_TIME,
    )
    raise AssertionError("Privilege amplification should fail")
except ValueError as exc:
    print("Blocked:", exc)

## 9. Revocation propagates through lineage

Short lifetime reduces exposure but is not immediate revocation. The server-side ledger rejects a child when its parent or any ancestor is revoked, and task closure invalidates all task grants.

In [ ]:
ledger.revoke(parent.grant_id)
child_request = AuthorizationRequest(
    operation_id="OP-CHILD-READ",
    audience="https://api.example.com/vendors",
    action="vendor:read",
    resource="department:data-ai",
    amount_cents=0,
    vendor_id="vendor-acme",
)
revoked = authorize_token(ledger, child_token, child_request, child_context, now=REFERENCE_TIME)
print(revoked.outcome, revoked.reason_codes)
assert revoked.outcome is DecisionOutcome.DENY
assert "grant_or_ancestor_revoked" in revoked.reason_codes

## 10. Standards and production authorization systems

RFC 8693 defines a token-exchange request/response protocol and actor/subject semantics. It does **not** define this lab's custom token, automatically make child authority narrower, propagate revocation, or provide atomic one-use consumption. The next cell shows only a request shape.

In [ ]:
exchange = token_exchange_request(
    subject_token="<authenticated-user-token>",
    actor_token="<attested-workload-token>",
    audience="procurement-api",
    scope="purchase_order:create",
)
print(exchange)
assert "access_token" not in exchange

Common production components solve different layers:

| Layer | Common choices | What to verify |
|---|---|---|
| Human identity | OIDC provider, passkeys/MFA | issuer, session, tenant, authentication age |
| Workload identity | SPIFFE/SPIRE, cloud workload identity | attestation, selectors, trust domain, rotation |
| Delegation/token | OAuth AS/STS, RFC 8693, RFC 9396, DPoP or mTLS | actor/subject, audience/resource, proof of possession, lifetime |
| Relationship authorization | OpenFGA | user + task + agent + resource bindings, tuple lifecycle |
| Contextual policy | Cedar/Verified Permissions, OPA/Rego | schema, policy version, amount/vendor/risk context |
| Enforcement and evidence | gateway/sidecar/library + transactional store + OpenTelemetry | fail-closed behavior, atomic consumption, outcome evidence |

OpenFGA and Cedar are complementary: relationship checks answer whether an agent/task is related to a resource; contextual policy evaluates the specific action and request context. Never infer success merely because an SDK call returned without raising—validate the decision and enforce it at the side-effect boundary.

In [ ]:
OPENFGA_MODEL = '''
model
  schema 1.1
type task
type agent
  relations
    define assigned_task: [task]
type resource
  relations
    define delegated: [task]
    define can_use: delegated
'''.strip()

CEDAR_POLICY = '''
permit (principal, action == Action::"CreatePurchaseOrder", resource)
when {
  context.taskId == "TASK-BUY-LAPTOPS-001" &&
  context.amountCents <= 500000 &&
  context.vendorApproved
};
'''.strip()

print(OPENFGA_MODEL)
print(CEDAR_POLICY)

These snippets are architecture illustrations, not evidence that a live OpenFGA or Cedar engine approved the request. A production integration test must load the exact model/schema and policy version, create scoped tuples/entities, evaluate positive and negative requests, and test cleanup and failure behavior.

## 11. Labelled evaluation

The evaluation population contains seven fixed cases: two legitimate operations (including the exact amount boundary) and five forbidden/lifecycle cases. `forbidden_allowed_count` is the safety error numerator over the five forbidden cases; `false_denial_count` is over the two legitimate cases.

In [ ]:
summary = run_evaluation()
print({
    "population": summary.case_count,
    "correct": summary.correct_count,
    "accuracy": str(summary.accuracy),
    "forbidden_population": summary.forbidden_case_count,
    "forbidden_allowed": summary.forbidden_allowed_count,
    "false_denials": summary.false_denial_count,
})
for row in summary.rows:
    print(row)

assert summary.correct_count == summary.case_count == 7
assert summary.forbidden_allowed_count == 0
assert summary.false_denial_count == 0

Seven deterministic cases prove the training invariants, not real-world security effectiveness. Production evaluation needs tenant and policy slices, adversarial identity/token cases, concurrent distributed consumers, key rotation, authorization-service outages, revocation latency, clock skew, and verified external outcomes.

## 12. Evidence inspection

Audit events retain identities, task/grant/operation/decision IDs, policy version, reason codes, and request/token digests—never the raw bearer token. The event is evidence of the PEP decision, not proof of an external side effect.

In [ ]:
context, _, token, ledger = build_demo_authority()
decision = authorize_token(ledger, token, demo_request(), context, now=REFERENCE_TIME)
event = ledger.audit_events[0]
print(event.model_dump(mode="json"))
assert token not in event.model_dump_json()
assert event.decision_id == decision.decision_id

## 13. Production upgrade path

| Training component | Production replacement | Required test |
|---|---|---|
| Fixed public Ed25519 test key | AS/STS with KMS/HSM, discovery, rotation | wrong/stale key, algorithm confusion, rollover |
| Synthetic workload evidence | SPIRE/cloud workload attestation | selector mismatch, trust-domain federation, expiry |
| In-memory directory | governed identity registry/IAM | disabled version, ownership transfer, cross-tenant access |
| In-process ledger lock | transactional shared store | multi-replica race, crash, retry, unknown outcome |
| Custom teaching claims | documented OAuth profile and AS policy | issuer/audience/resource, actor/subject, consent, DPoP/mTLS |
| Local revocation set | lifecycle service + short TTL + event propagation | ancestor revoke, task close, latency SLO, outage behavior |
| Decision audit | append-only protected telemetry | access control, retention, integrity, trace/outcome linkage |

Choose fail-closed behavior for consequential actions when identity, policy, revocation, or ledger state is unavailable. Define a separately risk-assessed degraded mode only for harmless read-only operations.

## 14. Exercises

1. **Implementation:** Add a new `supplier:email` action and prove that the root intent, grant, and request must all include it.
2. **Diagnosis:** Break one workload selector and explain which trust boundary rejects the request.
3. **Security:** Add a second child level, then prove the maximum delegation depth is enforced.
4. **Reliability:** Replace the in-process ledger with SQLite and design a transaction that preserves one-call consumption across two processes.
5. **Architecture:** Decide when OpenFGA alone is sufficient and when Cedar/OPA context is also required.
6. **Protocol:** Map the grant to RFC 9396 authorization details and evaluate sender-constraining it with DPoP (RFC 9449) or mTLS.
7. **Evaluation:** Add wrong-tenant, stale policy, key-rotation, and authorization-service outage cases; report explicit denominators.

## 15. Key takeaways

1. Human, logical-agent, and workload identities are distinct and must be bound from trusted state.
2. Authentication, consent, token scope, delegation, and runtime authorization solve different problems.
3. A child grant must attenuate every authority dimension, not only a list of permissions.
4. Signature validity does not establish current authorization; revocation, task state, resource context, and atomic consumption remain server-side concerns.
5. The model proposes an action. Trusted application controls validate, authorize, consume, execute, verify, and record it.

## Authoritative references

- NIST NCCoE, *Software and AI Agent Identity and Authorization* concept paper (2026)
- IETF RFC 8693, OAuth 2.0 Token Exchange
- IETF RFC 9700, Best Current Practice for OAuth 2.0 Security
- IETF RFC 8707, Resource Indicators for OAuth 2.0
- IETF RFC 8725, JWT Best Current Practices
- IETF RFC 9068, JWT Profile for OAuth 2.0 Access Tokens
- IETF RFC 9396, OAuth 2.0 Rich Authorization Requests
- IETF RFC 9449, OAuth 2.0 Demonstrating Proof of Possession
- SPIFFE specifications and SPIRE concepts
- OpenFGA, *Authorization for Agents* and *Task-Based Authorization*
- Cedar authorization documentation and security guidance

The 2026 WIMSE and AI-agent authorization documents discussed in the chapter are active Internet-Drafts, not published RFCs. Track their status before adopting draft-specific claims.